<a href="https://colab.research.google.com/github/rahmanullahkhan123/Generative_AI/blob/main/AI_Video_Generation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q diffusers transformers accelerate
!pip install -q imageio[ffmpeg]
!pip install -q opencv-python

In [19]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

PyTorch Version: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


In [20]:
import torch
import numpy as np
import imageio

from diffusers import DiffusionPipeline
from IPython.display import Video, display

In [21]:
model_id = "damo-vilab/text-to-video-ms-1.7b"


pipe = DiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
)


pipe = pipe.to("cuda")


print("Model Loaded Successfully")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Model Loaded Successfully


In [22]:
prompt = """
A futuristic city at night,
flying cars in the sky,
neon lights,
cinematic camera movement,
realistic style,
high quality
"""


video_output = pipe(
    prompt,
    num_inference_steps=25
)


print("Video Generated")

  0%|          | 0/25 [00:00<?, ?it/s]

Video Generated


In [23]:
frames = video_output.frames


print(type(frames))
print(len(frames))

<class 'numpy.ndarray'>
1


In [24]:
# Remove batch dimension
if isinstance(frames[0], list):
    frames = frames[0]


print("Number of frames:", len(frames))
print("Frame type:", type(frames[0]))

Number of frames: 1
Frame type: <class 'numpy.ndarray'>


In [41]:
import numpy as np

print("processed_frames type:")
print(type(processed_frames))

print("\nFull shape:")
print(np.array(processed_frames).shape)

print("\nFirst element shape:")
print(np.array(processed_frames[0]).shape)

processed_frames type:
<class 'list'>

Full shape:
(1, 16, 256, 256, 3)

First element shape:
(16, 256, 256, 3)


In [42]:
import numpy as np
import imageio


frames = np.array(processed_frames)

print("Original:", frames.shape)


# Remove unnecessary dimensions
frames = np.squeeze(frames)

print("After squeeze:", frames.shape)


# If frames are (C,T,H,W) convert to (T,H,W,C)
if frames.ndim == 4 and frames.shape[0] in [1,3,4]:

    frames = np.transpose(frames, (1,2,3,0))


# If frames are (T,C,H,W) convert to (T,H,W,C)
elif frames.ndim == 4 and frames.shape[1] in [1,3,4]:

    frames = np.transpose(frames, (0,2,3,1))


print("After transpose:", frames.shape)


# Convert float to uint8
if frames.dtype != np.uint8:

    frames = np.clip(frames,0,1)
    frames = (frames*255).astype(np.uint8)


# Verify every frame
for i, frame in enumerate(frames):

    if frame.shape[-1] not in [1,3,4]:
        raise ValueError(
            f"Frame {i} has invalid shape {frame.shape}"
        )


imageio.mimsave(
    "generated_video.mp4",
    frames,
    fps=8
)


print("Video saved successfully!")

Original: (1, 16, 256, 256, 3)
After squeeze: (16, 256, 256, 3)
After transpose: (16, 256, 256, 3)
Video saved successfully!


In [43]:
display(
    Video(
        video_path,
        embed=True
    )
)

In [47]:
import gradio as gr

print("Gradio imported successfully")

Gradio imported successfully


In [49]:
import torch
import gradio as gr
import numpy as np
import imageio

from diffusers import DiffusionPipeline

In [51]:
demo = gr.Interface(

    fn=generate_video,

    inputs=[
        gr.Textbox(
            label="Enter Video Prompt",
            placeholder="A robot walking in a futuristic city"
        )
    ],

    outputs=[
        gr.Video(
            label="Generated AI Video"
        )
    ],

    title="AI Video Generation System",

    description="Generate videos from text using AI."
)


demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ea49d4ea4a748a4e8c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
